# Multi-Node Distributed Computing with SLURM

In this notebook we move beyond a single machine. Using **dask-jobqueue** we launch
Dask workers as SLURM jobs so that our analysis spreads across multiple compute nodes
connected by InfiniBand.

In [ ]:
from pathlib import Path
import time

import dask.array as da
import dask.dataframe as dd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline

## Data path

We use the same lv2 scientific dataset as in the previous notebook.

In [ ]:
lv2_path = Path('/net/pr2/projects/tutorial/2026-04-15-hpda/dataset/lv2/data/')
assert lv2_path.exists(), f"Data path not found: {lv2_path}"
print(f"Data path OK: {lv2_path}")

## Setting up a SLURMCluster

`SLURMCluster` from **dask-jobqueue** submits SLURM jobs that each become a Dask
worker. Key parameters:

| Parameter | Value | Why |
|-----------|-------|-----|
| `cores` | 4 | CPU cores per worker |
| `memory` | `"64GB"` | RAM per worker (16 GB/core) |
| `account` | `"tutorial"` | SLURM billing account |
| `walltime` | `"00:30:00"` | Max job duration |
| `interface` | `"ib0"` | InfiniBand for fast inter-node traffic |
| `queue` | `"tutorial"` | SLURM partition |

In [ ]:
from dask_jobqueue import SLURMCluster

cluster = SLURMCluster(
    cores=4,
    memory="64GB",
    account="tutorial",
    walltime="00:30:00",
    interface="ib0",
    queue="tutorial",
)

### Inspect the generated SLURM script

`job_script()` shows the exact `#SBATCH` script that will be submitted for each
worker. Check that the resource requests match your expectations.

In [ ]:
print(cluster.job_script())

### Scale the cluster

`cluster.scale(n)` submits *n* SLURM jobs. Each job starts one Dask worker.
Workers may take 1-2 minutes to appear in the queue and connect.

In [ ]:
cluster.scale(2)

### Connect a Client

The `Client` connects our notebook to the cluster scheduler. It also provides a
**dashboard** link — open it in a new browser tab to monitor workers, tasks, and memory.

> **Port collision**: If you set up a participant number in `00_setup.ipynb`, uncomment
> the `dashboard_address` line below to use your unique port.

#### Accessing the Dask Dashboard via SSH Tunnel

On HPC systems like Athena, compute jobs run on **worker nodes** (e.g. `t0006`) that are on an internal network — they are **not directly accessible** from the outside. The only publicly reachable machine is the **login node** (`athena.cyfronet.pl`), which acts as a gateway.

```
[Your laptop] ---internet---> [Login node: athena.cyfronet.pl] ---InfiniBand---> [Worker node: t0006]
```

Because the Dask dashboard runs on the worker node (port 8788), you cannot simply open `http://t0006:8788` from your laptop. Instead, you create an **SSH tunnel** through the login node:

```bash
ssh -L 8788:t0006:8788 tutorial249@athena.cyfronet.pl
```

This command:
1. Connects you to the login node (`athena.cyfronet.pl`)
2. Tells SSH to forward your local port `8788` to `t0006:8788` through the login node
3. After entering your password, you can open **`http://localhost:8788`** in your local browser to see the Dask dashboard

> **Note:** Replace `t0006` with the actual worker node assigned to your job (check with `hostname` on the compute node) and `tutorial249` with your username. The tunnel must stay open for the entire session.

In [ ]:
from dask.distributed import Client

# Uncomment and adjust if you assigned a participant number in 00_setup:
# participant_number = 1
# dashboard_port = 8787 + participant_number

client = Client(cluster)
# client = Client(cluster, dashboard_address=f":{dashboard_port}")
client

Wait until workers are running before continuing. The cell below blocks until
at least 1 worker is available (timeout 120 s).

In [ ]:
client.wait_for_workers(n_workers=1, timeout=120)
print(f"Workers ready: {len(client.scheduler_info()['workers'])}")

### Verify InfiniBand is in use

The `SLURMCluster` was configured with `interface="ib0"`, which tells each Dask worker to bind its communications socket to the InfiniBand interface. We can confirm this by inspecting the worker addresses reported by the scheduler.

From `ip a` on the compute nodes, the InfiniBand subnets are `172.23.16.0/20` through `172.23.19.0/20`. If worker addresses fall in this range, inter-worker traffic travels over InfiniBand — not the 1 GbE Ethernet (`eth0`/`eth1`, which are `DOWN`).

In [ ]:
import ipaddress

IB_SUBNETS = [
    ipaddress.ip_network("172.23.16.0/20"),
    ipaddress.ip_network("172.23.17.0/20"),
    ipaddress.ip_network("172.23.18.0/20"),
    ipaddress.ip_network("172.23.19.0/20"),
]

info = client.scheduler_info()
print(f"Scheduler address : {info['address']}")
print()
for addr, w in info['workers'].items():
    host = w['host']
    try:
        ip = ipaddress.ip_address(host)
        on_ib = any(ip in net for net in IB_SUBNETS)
        ib_tag = "✓ InfiniBand" if on_ib else "✗ NOT InfiniBand"
    except ValueError:
        ib_tag = "(hostname — check with `nslookup`)"
    print(f"  Worker {addr}")
    print(f"    host={host}  {ib_tag}")
    print(f"    threads={w['nthreads']}  memory={w['memory_limit']/1e9:.1f} GB")

## Loading data across the cluster

Dask reads the Parquet dataset lazily. Metadata (column names, types, partition
boundaries) is fetched immediately; actual bytes are read only when `.compute()` or
`.persist()` is called.

In [ ]:
ddf = dd.read_parquet(lv2_path)
print(f"Columns : {list(ddf.columns)}")
print(f"Partitions: {ddf.npartitions}")

In [ ]:
# Estimate total dataset volume without reading all data.
# sample one partition to get bytes-per-row, then extrapolate.
sample = ddf.get_partition(0).compute()
rows_per_partition = len(sample)
bytes_per_row = sample.memory_usage(deep=True).sum() / max(rows_per_partition, 1)
estimated_total_gb = bytes_per_row * rows_per_partition * ddf.npartitions / 1e9

print(f"Partitions       : {ddf.npartitions}")
print(f"Rows in part 0   : {rows_per_partition:,}")
print(f"Bytes / row      : {bytes_per_row:.1f}")
print(f"Part 0 in RAM    : {rows_per_partition * bytes_per_row / 1e6:.1f} MB")
print(f"Estimated total  : {estimated_total_gb:.2f} GB  (assuming uniform partition size)")

In [ ]:
ddf.head()

### Filter, select columns, and persist

We apply three optimisation steps:

1. **Filter** — keep only the `0p5nA` dataset (reduces volume)
2. **Column selection** — keep only the columns we need
3. **Repartition** — redistribute into 1 000 evenly-sized partitions for better load-balancing
4. **Persist** — trigger computation and keep the result in cluster RAM so subsequent
   operations are fast

### Understanding the "Sending large graph" warning

You may see a warning like:

```
UserWarning: Sending large graph of size 75.33 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly …
```

**What it means.**  
Before any `.compute()` or `.persist()`, Dask serialises the entire *task graph* — the DAG of instructions describing what to read, filter, and compute — and sends it to the scheduler, which forwards relevant pieces to each worker. With thousands of Parquet partitions, even the task *descriptions* (file paths, read parameters, schema bytes, filter predicates …) can add up to tens of megabytes. **No actual data is sent at this point** — workers read directly from the parallel filesystem (`/net/pr2/…`).

**When it matters.**  
A 75 MB graph takes a second or two to serialise and transmit — a one-time overhead before computation starts. For a job that then reads hundreds of gigabytes from disk, this is negligible.

**Why does it happen here?**  
`dd.read_parquet` on a large dataset with many files creates one task per Parquet row-group. Each task embeds the file path and Parquet metadata. This is the correct way to load data with Dask; the warning is informational, not an error.

**After `.persist()`.**  
Once `df` is persisted, all subsequent operations (`.mean()`, `.histogram()`, …) receive only a tiny graph of *Future* references — the per-partition data already lives in worker RAM, so no large graph is sent again.

In [ ]:
cols = ['dataset', 'channel_no', 'trc_file_no', 'sign',
        'peak_length_ns', 'peak_amplitude_mV']

df = (
    ddf
    .query('dataset == "0p5nA"')[cols]
    .repartition(npartitions=1000)
    .persist()
)

In [ ]:
# Quick sanity check — mean peak amplitude
df.peak_amplitude_mV.mean().compute()

### Distributed histogram

The histogram is computed in parallel across all workers. Each worker processes its
local partitions and sends partial counts to the scheduler for aggregation.

In [ ]:
%%time
hist_vals, edges = da.histogram(
    df.query('channel_no == 0 and sign == "negative" and peak_length_ns > 0.4')
      .peak_amplitude_mV,
    bins=200,
    range=(0, 200),
)
hist_computed = hist_vals.compute()

In [ ]:
plt.plot(edges[:-1], hist_computed, drawstyle='steps-post')
plt.yscale('log')
plt.xlabel('Peak amplitude [mV]')
plt.ylabel('Counts')
plt.title('0p5nA · ch 0 · negative · peak_length > 0.4 ns')
plt.grid(True, alpha=0.3)
plt.tight_layout()

## Scaling experiment: workers vs wall time

Does adding more workers make the computation faster? Let us measure the histogram
wall time at 1, 2, and 4 workers.

> **Note**: `cluster.scale()` may take a minute each time new workers are
> allocated by SLURM. Be patient.

In [ ]:
def time_histogram(client_obj, df, n_runs=3):
    """Return median wall time (seconds) for the histogram computation."""
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        da.histogram(
            df.query('channel_no == 0 and sign == "negative" and peak_length_ns > 0.4')
              .peak_amplitude_mV,
            bins=200,
            range=(0, 200),
        )[0].compute()
        times.append(time.perf_counter() - t0)
    return float(np.median(times))

In [ ]:
worker_counts = [1, 2, 4]
timings = []

for n in worker_counts:
    cluster.scale(n)
    client.wait_for_workers(n_workers=n, timeout=180)
    # Allow a moment for the cluster to stabilise
    time.sleep(5)

    t = time_histogram(client, df)
    timings.append(t)
    print(f"Workers: {n}  |  Median time: {t:.2f} s")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(worker_counts, timings, 'o-', linewidth=2, markersize=8)
plt.xlabel('Number of SLURM workers')
plt.ylabel('Wall time [s]')
plt.title('Histogram computation — scaling')
plt.xticks(worker_counts)
plt.grid(True, alpha=0.3)
plt.tight_layout()

In [ ]:
# Compute speedup relative to 1 worker
speedups = [timings[0] / t for t in timings]
for n, s in zip(worker_counts, speedups):
    print(f"Workers: {n}  |  Speedup: {s:.2f}x")

## Bonus: synthetic timeseries at scale

`dd.demo.make_timeseries` generates a large random DataFrame partitioned by month.
Useful for quick distributed experiments without needing real data on disk.

In [ ]:
ts = dd.demo.make_timeseries(
    start="2000-01-01",
    end="2024-12-31",
    dtypes={"x": float, "y": float, "z": float},
    freq="1s",
    partition_freq="1ME",
    seed=42,
)
print(f"Partitions: {ts.npartitions}")
ts.head()

In [ ]:
%%time
daily_mean = ts["x"].resample("1D").mean().compute()

In [ ]:
daily_mean.plot(figsize=(10, 3), alpha=0.7)
plt.xlabel("Date")
plt.ylabel("Daily mean of x")
plt.title("25 years of synthetic data — daily mean")
plt.tight_layout()

## Cleanup

Always close the client and cluster when you are done to release SLURM jobs.

In [ ]:
client.close()
cluster.close()

## Exercise: scaling study

Go back to the scaling experiment section and extend it:

1. Scale to **1, 2, 4, 8** workers (if the queue allows).
2. Time the histogram computation at each scale.
3. Compute the **parallel efficiency**: $E = \frac{S}{n}$ where $S$ is the speedup
   and $n$ is the number of workers. Perfect scaling gives $E = 1$.
4. Plot both speedup and efficiency vs number of workers.

**Scaffolding**:

In [ ]:
# ── Exercise scaffolding ──

worker_counts_ext = [1, 2, 4, 8]
timings_ext = []

# Recreate cluster and client
# cluster = SLURMCluster(cores=4, memory="64GB", account="tutorial",
#                        walltime="00:30:00", interface="ib0", queue="tutorial")
# client = Client(cluster)

for n in worker_counts_ext:
    # TODO: scale cluster, wait for workers, measure time
    pass

# TODO: compute speedups and efficiencies
# speedups_ext = ___
# efficiencies_ext = ___

# TODO: create a 1×2 subplot — left: speedup, right: efficiency
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
# ...

# Remember to close!
# client.close()
# cluster.close()

**Hints**:
- Reuse the `time_histogram()` function defined above.
- If the queue does not have capacity for 8 workers, stop at 4 and note it.
- Perfect linear speedup is rarely achieved — inter-node communication overhead and
  data redistribution eat into the gains.